In [1]:
!pip install pypower



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
from pypower.api import case30, runpf, ppoption
from pprint import pprint

def run_newton_raphson_power_flow():
    """
    This function performs a Newton-Raphson power flow analysis on the
    IEEE 30-bus test system using the PYPOWER library.
    """

    # --- 1. Load System Data ---
    # PYPOWER comes with several standard test cases built-in.
    # We will load the IEEE 30-bus case data.
    # This data is stored in a dictionary and contains all the necessary
    # information about the power system network.
    ppc = case30()

    # The 'ppc' dictionary contains the following keys:
    # 'bus': Bus data (bus number, type, loads, voltage, etc.)
    # 'gen': Generator data (location, active/reactive power output, limits, etc.)
    # 'branch': Branch data (from/to bus, resistance, reactance, etc.)
    # 'version': The case format version
    # 'baseMVA': The system's base power rating in MVA (Mega Volt-Amperes)

    print("--- IEEE 30-Bus System Data Loaded ---")
    print(f"System Base MVA: {ppc['baseMVA']}")
    print(f"Number of buses: {len(ppc['bus'])}")
    print(f"Number of generators: {len(ppc['gen'])}")
    print(f"Number of transmission lines: {len(ppc['branch'])}")
    print("-" * 40)


    # --- 2. Set Power Flow Options ---
    # We can specify the algorithm and output options for the power flow solver.
    # By default, runpf uses the Newton-Raphson method.
    # We will explicitly define the options for clarity.
    # 'PF_ALG = 1' corresponds to the Newton-Raphson method.
    # 'VERBOSE = 1' will print summary information about the solution.
    # 'OUT_ALL = 0' will suppress detailed output to the console. We will print our own summary.
    ppopt = ppoption(PF_ALG=1, VERBOSE=1, OUT_ALL=0)


    # --- 3. Run the Power Flow Solver ---
    # The runpf() function takes the case data and options as input
    # and returns the solved case.
    # The results dictionary will contain the calculated power flow solution,
    # including bus voltages, angles, power injections, and flows.
    print("\n--- Running Newton-Raphson Power Flow ---")
    results = runpf(ppc, ppopt)[0] # runpf returns a tuple (results, success_flag)
    print("Power flow calculation complete.")
    print("-" * 40)


    # --- 4. Display Results ---
    if results['success']:
        print("\n--- Power Flow Analysis Results ---")
        print("Convergence achieved successfully!")

        # The results are stored in the 'bus', 'gen', and 'branch' fields
        # of the 'results' dictionary. The values are now the solved values.

        # We will print a formatted table of the bus voltage results.
        # The columns in the results['bus'] matrix are:
        # 0: bus number
        # 7: voltage magnitude (in per-unit, p.u.)
        # 8: voltage angle (in degrees)
        # 2: real power demand (Pd)
        # 3: reactive power demand (Qd)

        print("\nBus Voltage Magnitudes and Angles:")
        print("-------------------------------------------------")
        print("Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)")
        print("-------------------------------------------------")
        for bus in results['bus']:
            bus_num = int(bus[0])
            vm_pu = bus[7]
            va_deg = bus[8]
            pd_mw = bus[2]
            qd_mvar = bus[3]
            print(f"{bus_num:<8}| {vm_pu:<16.4f}| {va_deg:<13.4f}| {pd_mw:<11.2f}| {qd_mvar:<12.2f}")
        print("-------------------------------------------------")


        # The 'gen' matrix contains the dispatch of each generator.
        # 1: Real power output (Pg) in MW
        # 2: Reactive power output (Qg) in MVAr
        print("\nGenerator Dispatch:")
        print("------------------------------------------")
        print("Gen on Bus | P_gen (MW)   | Q_gen (MVAr)")
        print("------------------------------------------")
        for gen in results['gen']:
             bus_num = int(gen[0])
             pg_mw = gen[1]
             qg_mvar = gen[2]
             print(f"{bus_num:<11}| {pg_mw:<12.2f}| {qg_mvar:<12.2f}")
        print("------------------------------------------")


    else:
        print("\n--- Power Flow Analysis Failed ---")
        print("The Newton-Raphson algorithm did not converge.")


if __name__ == '__main__':
    run_newton_raphson_power_flow()



--- IEEE 30-Bus System Data Loaded ---
System Base MVA: 100.0
Number of buses: 30
Number of generators: 6
Number of transmission lines: 41
----------------------------------------

--- Running Newton-Raphson Power Flow ---
PYPOWER Version 5.1.18, 10-Apr-2025 -- AC Power Flow (Newton)


Newton's method power flow converged in 3 iterations.
Power flow calculation complete.
----------------------------------------

--- Power Flow Analysis Results ---
Convergence achieved successfully!

Bus Voltage Magnitudes and Angles:
-------------------------------------------------
Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)
-------------------------------------------------
1       | 1.0000          | 0.0000       | 0.00       | 0.00        
2       | 1.0000          | -0.4155      | 21.70      | 12.70       
3       | 0.9831          | -1.5221      | 2.40       | 1.20        
4       | 0.9801          | -1.7947      | 7.60       | 1.60        
5       | 0.9824          | -1

In [5]:
# First, you need to install the PYPOWER library.
# You can do this by running the following command in your terminal:
# pip install pypower

import numpy as np
from pypower.api import runpf, ppoption
from pprint import pprint

def create_custom_30_bus_system():
    """
    This function creates a custom PYPOWER case (ppc) for a 30-bus system.
    The data has been populated from the user-provided image (Tables A1 and A2).
    """
    ppc = {
        "version": '2',
        "baseMVA": 100.0,
    }

    # --- Bus Data (from Table A1) ---
    # Bus codes from the table are mapped to PYPOWER standard:
    # Table Code 1 (Slack) -> PYPOWER Type 3
    # Table Code 2 (PV)    -> PYPOWER Type 2
    # Table Code 0 (PQ)    -> PYPOWER Type 1
    #
    # Columns:
    # 0: bus_i, 1: type, 2: Pd, 3: Qd, 4: Gs, 5: Bs, 6: area, 7: Vm,
    # 8: Va, 9: baseKV, 10: zone, 11: Vmax, 12: Vmin
    ppc['bus'] = np.array([
        [1,  3,   0.0,    0.0,    0.0, 0.0, 1, 1.06,  0.0, 132, 1, 1.1, 0.9],
        [2,  2,  21.7,   12.7,    0.0, 0.0, 1, 1.043, 0.0, 132, 1, 1.1, 0.9],
        [3,  1,   2.4,    1.2,    0.0, 0.0, 1, 1.0,   0.0, 132, 1, 1.1, 0.9],
        [4,  1,   7.6,    1.6,    0.0, 0.0, 1, 1.0,   0.0, 132, 1, 1.1, 0.9],
        [5,  2,  94.2,   19.0,    0.0, 0.0, 1, 1.01,  0.0, 132, 1, 1.1, 0.9],
        [6,  1,   0.0,    0.0,    0.0, 0.0, 1, 1.0,   0.0, 132, 1, 1.1, 0.9],
        [7,  1,  22.8,   10.9,    0.0, 0.0, 1, 1.0,   0.0, 132, 1, 1.1, 0.9],
        [8,  2,  30.0,   30.0,    0.0, 0.0, 1, 1.01,  0.0, 132, 1, 1.1, 0.9],
        [9,  1,   0.0,    0.0,    0.0, 0.0, 1, 1.0,   0.0, 132, 1, 1.1, 0.9],
        [10, 1,   5.8,    2.0,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [11, 2,   0.0,    0.0,    0.0, 0.0, 1, 1.082, 0.0,  33, 1, 1.1, 0.9],
        [12, 1,  11.2,    7.5,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [13, 2,   0.0,    0.0,    0.0, 0.0, 1, 1.071, 0.0,  33, 1, 1.1, 0.9],
        [14, 1,   6.2,    1.6,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [15, 1,   8.2,    2.5,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [16, 1,   3.5,    1.8,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [17, 1,   9.0,    5.8,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [18, 1,   3.2,    0.9,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [19, 1,   9.5,    3.4,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [20, 1,   2.2,    0.7,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [21, 1,  17.5,   11.2,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [22, 1,   0.0,    0.0,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [23, 1,   3.2,    1.6,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [24, 1,   8.7,    6.7,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [25, 1,   0.0,    0.0,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [26, 1,   3.5,    2.3,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [27, 1,   0.0,    0.0,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [28, 1,   0.0,    0.0,    0.0, 0.0, 1, 1.0,   0.0, 132, 1, 1.1, 0.9],
        [29, 1,   2.4,    0.9,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9],
        [30, 1,  10.6,    1.9,    0.0, 0.0, 1, 1.0,   0.0,  33, 1, 1.1, 0.9]
    ])

    # --- Generator Data (from Table A1) ---
    # Columns:
    # 0: bus, 1: Pg, 2: Qg, 3: Qmax, 4: Qmin, 5: Vg,
    # 6: mBase, 7: status, 8: Pmax, 9: Pmin
    ppc['gen'] = np.array([
        [1,  138.59, -15.97, 100, -30, 1.06,  100, 1, 360, 0],
        [2,   57.56,  -1.70, 100, -30, 1.043, 100, 1, 140, 0],
        [5,   37.00,  27.30, 100, -30, 1.01,  100, 1, 100, 0],
        [8,   37.30,  22.40, 100, -30, 1.01,  100, 1, 100, 0],
        [11,  17.91,  16.20, 100, -30, 1.082, 100, 1,  50, 0],
        [13,  16.39,  10.90, 100, -30, 1.071, 100, 1,  50, 0]
    ])

    # --- Branch Data (from Table A2) ---
    # The B/2 column is used for the total line charging susceptance 'b'.
    # Columns:
    # 0: fbus, 1: tbus, 2: r, 3: x, 4: b, 5: rateA,
    # 6: rateB, 7: rateC, 8: ratio, 9: angle, 10: status
    ppc['branch'] = np.array([
        [1,  2,  0.0192, 0.0575, 0.0264, 130, 130, 130, 0, 0, 1],
        [1,  7,  0.0452, 0.1652, 0.0204, 130, 130, 130, 0, 0, 1],
        [2,  8,  0.0570, 0.1737, 0.0184, 65,   65,  65, 0, 0, 1],
        [3,  9,  0.0132, 0.0379, 0.0042, 130, 130, 130, 0, 0, 1],
        [8,  4,  0.0472, 0.1983, 0.0209, 130, 130, 130, 0, 0, 1],
        [9,  5,  0.0581, 0.1763, 0.0187, 65,   65,  65, 0, 0, 1],
        [2,  6,  0.0119, 0.0414, 0.0045, 90,   90,  90, 0, 0, 1],
        [9,  8,  0.0460, 0.1160, 0.0102, 70,   70,  70, 0, 0, 1],
        [3,  10, 0.0267, 0.0820, 0.0085, 130, 130, 130, 0, 0, 1],
        [9,  11, 0.0120, 0.0420, 0.0045, 32,   32,  32, 0, 0, 1],
        [9,  12, 0.0,    0.2080, 0.0,    65,   65,  65, 0, 0, 1],
        [11, 1,  0.0,    0.5560, 0.0,    32,   32,  32, 0, 0, 1],
        [12, 9,  0.0,    0.2080, 0.0,    65,   65,  65, 0, 0, 1],
        [8,  13, 0.0,    0.1100, 0.0,    65,   65,  65, 0, 0, 1],
        [13, 6,  0.0,    0.2560, 0.0,    65,   65,  65, 0, 0, 1],
        [13, 14, 0.0,    0.1400, 0.0,    65,   65,  65, 0, 0, 1],
        [13, 15, 0.1231, 0.2559, 0.0,    32,   32,  32, 0, 0, 1],
        [13, 16, 0.0662, 0.1304, 0.0,    32,   32,  32, 0, 0, 1],
        [15, 14, 0.0945, 0.1987, 0.0,    32,   32,  32, 0, 0, 1],
        [14, 15, 0.2210, 0.1997, 0.0,    16,   16,  16, 0, 0, 1],
        [16, 17, 0.0824, 0.1923, 0.0,    16,   16,  16, 0, 0, 1],
        [15, 18, 0.1073, 0.2185, 0.0,    16,   16,  16, 0, 0, 1],
        [18, 19, 0.0639, 0.1292, 0.0,    16,   16,  16, 0, 0, 1],
        [19, 20, 0.0340, 0.0680, 0.0,    32,   32,  32, 0, 0, 1],
        [12, 17, 0.0324, 0.0845, 0.0,    32,   32,  32, 0, 0, 1],
        [12, 21, 0.0348, 0.0749, 0.0,    32,   32,  32, 0, 0, 1],
        [12, 22, 0.0727, 0.1499, 0.0,    32,   32,  32, 0, 0, 1],
        [21, 22, 0.0116, 0.0236, 0.0,    32,   32,  32, 0, 0, 1],
        [15, 23, 0.1000, 0.2020, 0.0,    16,   16,  16, 0, 0, 1],
        [22, 24, 0.1150, 0.1790, 0.0,    16,   16,  16, 0, 0, 1],
        [23, 24, 0.1320, 0.2700, 0.0,    16,   16,  16, 0, 0, 1],
        [24, 25, 0.1885, 0.3292, 0.0,    16,   16,  16, 0, 0, 1],
        [25, 26, 0.2544, 0.3800, 0.0,    16,   16,  16, 0, 0, 1],
        [25, 27, 0.1093, 0.2087, 0.0,    16,   16,  16, 0, 0, 1],
        [27, 28, 0.0,    0.3960, 0.0,    65,   65,  65, 0, 0, 1],
        [27, 29, 0.2198, 0.4153, 0.0,    16,   16,  16, 0, 0, 1],
        [27, 30, 0.3202, 0.6027, 0.0,    16,   16,  16, 0, 0, 1],
        [29, 30, 0.2399, 0.4533, 0.0,    16,   16,  16, 0, 0, 1],
        [9,  28, 0.0169, 0.0599, 0.0065, 32,   32,  32, 0, 0, 1],
        [4,  28, 0.0636, 0.2000, 0.0214, 32,   32,  32, 0, 0, 1],
    ])

    return ppc


def run_newton_raphson_power_flow():
    """
    This function performs a Newton-Raphson power flow analysis on a
    custom 30-bus test system using the PYPOWER library.
    """

    # --- 1. Load Custom System Data ---
    # We call our function to create the custom case data structure.
    ppc = create_custom_30_bus_system()

    print("--- Custom 30-Bus System Data Loaded ---")
    print(f"System Base MVA: {ppc['baseMVA']}")
    print(f"Number of buses: {len(ppc['bus'])}")
    print(f"Number of generators: {len(ppc['gen'])}")
    print(f"Number of transmission lines: {len(ppc['branch'])}")
    print("-" * 40)


    # --- 2. Set Power Flow Options ---
    # 'PF_ALG = 1' corresponds to the Newton-Raphson method.
    # 'VERBOSE = 1' will print summary information about the solution.
    # 'OUT_ALL = 0' suppresses detailed output to the console.
    ppopt = ppoption(PF_ALG=1, VERBOSE=1, OUT_ALL=0)


    # --- 3. Run the Power Flow Solver ---
    # The runpf() function takes the case data and options as input
    # and returns the solved case.
    print("\n--- Running Newton-Raphson Power Flow ---")
    results, success = runpf(ppc, ppopt)
    print("Power flow calculation complete.")
    print("-" * 40)


    # --- 4. Display Results ---
    if success:
        print("\n--- Power Flow Analysis Results ---")
        print("Convergence achieved successfully!")

        print("\nBus Voltage Magnitudes and Angles:")
        print("-------------------------------------------------")
        print("Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)")
        print("-------------------------------------------------")
        for bus in results['bus']:
            bus_num = int(bus[0])
            vm_pu = bus[7]
            va_deg = bus[8]
            pd_mw = bus[2]
            qd_mvar = bus[3]
            print(f"{bus_num:<8}| {vm_pu:<16.4f}| {va_deg:<13.4f}| {pd_mw:<11.2f}| {qd_mvar:<12.2f}")
        print("-------------------------------------------------")

        print("\nGenerator Dispatch:")
        print("------------------------------------------")
        print("Gen on Bus | P_gen (MW)   | Q_gen (MVAr)")
        print("------------------------------------------")
        for gen in results['gen']:
             bus_num = int(gen[0])
             pg_mw = gen[1]
             qg_mvar = gen[2]
             print(f"{bus_num:<11}| {pg_mw:<12.2f}| {qg_mvar:<12.2f}")
        print("------------------------------------------")


    else:
        print("\n--- Power Flow Analysis Failed ---")
        print("The Newton-Raphson algorithm did not converge.")


if __name__ == '__main__':
    run_newton_raphson_power_flow()


--- Custom 30-Bus System Data Loaded ---
System Base MVA: 100.0
Number of buses: 30
Number of generators: 6
Number of transmission lines: 40
----------------------------------------

--- Running Newton-Raphson Power Flow ---
PYPOWER Version 5.1.18, 10-Apr-2025 -- AC Power Flow (Newton)


Newton's method power flow converged in 3 iterations.
Power flow calculation complete.
----------------------------------------

--- Power Flow Analysis Results ---
Convergence achieved successfully!

Bus Voltage Magnitudes and Angles:
-------------------------------------------------
Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)
-------------------------------------------------
1       | 1.0600          | 0.0000       | 0.00       | 0.00        
2       | 1.0430          | -1.9111      | 21.70      | 12.70       
3       | 1.0392          | -11.2491     | 2.40       | 1.20        
4       | 1.0197          | -9.7513      | 7.60       | 1.60        
5       | 1.0100          | 

In [9]:
# First, you need to install the PYPOWER library.
# You can do this by running the following command in your terminal:
# pip install pypower

import numpy as np
from pypower.api import runpf, ppoption
from pprint import pprint

def create_custom_30_bus_system():
    """
    This function creates a custom PYPOWER case (ppc) for a 30-bus system,
    populated with the user-provided busdata and linedata.
    """
    ppc = {
        "version": '2',
        "baseMVA": 100.0,
    }

    # --- Bus Data ---
    # Data is extracted and formatted from the user's 'busdata' matrix.
    # Bus type mapping: 1 -> 3 (Slack), 2 -> 2 (PV), 3 -> 1 (PQ)
    busdata = np.array([
        [1,  1, 1.06,  0,   0,   0,    0,    0,   0,   0],
        [2,  2, 1.043, 0,  40,  50.0, 21.7, 12.7, -40, 50],
        [3,  3, 1.0,   0,   0,   0,   2.4,  1.2,   0,   0],
        [4,  3, 1.06,  0,   0,   0,   7.6,  1.6,   0,   0],
        [5,  2, 1.01,  0,   0,  37.0, 94.2, 19.0, -40, 40],
        [6,  3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [7,  3, 1.0,   0,   0,   0,  22.8, 10.9,   0,   0],
        [8,  2, 1.01,  0,   0,  37.3, 30.0, 30.0, -10, 40],
        [9,  3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [10, 3, 1.0,   0,   0,  19.0,  5.8,  2.0,   0,   0],
        [11, 2, 1.082, 0,   0,  16.2,  0.0,  0.0,  -6,  24],
        [12, 3, 1.0,   0,   0,   0,  11.2,  7.5,   0,   0],
        [13, 2, 1.071, 0,   0,  10.6,  0.0,  0.0,  -6,  24],
        [14, 3, 1.0,   0,   0,   0,   6.2,  1.6,   0,   0],
        [15, 3, 1.0,   0,   0,   0,   8.2,  2.5,   0,   0],
        [16, 3, 1.0,   0,   0,   0,   3.5,  1.8,   0,   0],
        [17, 3, 1.0,   0,   0,   0,   9.0,  5.8,   0,   0],
        [18, 3, 1.0,   0,   0,   0,   3.2,  0.9,   0,   0],
        [19, 3, 1.0,   0,   0,   0,   9.5,  3.4,   0,   0],
        [20, 3, 1.0,   0,   0,   0,   2.2,  0.7,   0,   0],
        [21, 3, 1.0,   0,   0,   0,  17.5, 11.2,   0,   0],
        [22, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [23, 3, 1.0,   0,   0,   0,   3.2,  1.6,   0,   0],
        [24, 3, 1.0,   0,   0,  4.3,  8.7,  6.7,   0,   0],
        [25, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [26, 3, 1.0,   0,   0,   0,   3.5,  2.3,   0,   0],
        [27, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [28, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [29, 3, 1.0,   0,   0,   0,   2.4,  0.9,   0,   0],
        [30, 3, 1.0,   0,   0,   0,  10.6,  1.9,   0,   0]
    ])

    # PYPOWER bus matrix
    # Columns: bus_i, type, Pd, Qd, Gs, Bs, area, Vm, Va, baseKV, zone, Vmax, Vmin
    ppc['bus'] = np.zeros((30, 13))
    for i in range(30):
        bus_type = busdata[i, 1]
        if bus_type == 1:
            ppc['bus'][i, 1] = 3  # Slack
        elif bus_type == 2:
            ppc['bus'][i, 1] = 2  # PV
        else:
            ppc['bus'][i, 1] = 1  # PQ

        ppc['bus'][i, 0] = busdata[i, 0]    # Bus Number
        ppc['bus'][i, 2] = busdata[i, 6]    # Pd
        ppc['bus'][i, 3] = busdata[i, 7]    # Qd
        ppc['bus'][i, 7] = busdata[i, 2]    # Vm
        ppc['bus'][i, 8] = busdata[i, 3]    # Va
        # Set default values for other fields
        ppc['bus'][i, 6] = 1                # area
        ppc['bus'][i, 9] = 132              # baseKV
        ppc['bus'][i, 10] = 1               # zone
        ppc['bus'][i, 11] = 1.1             # Vmax
        ppc['bus'][i, 12] = 0.9             # Vmin

    # Special handling for shunts on buses 10 and 24 from busdata
    ppc['bus'][9, 5] = busdata[9, 5]     # Shunt Bs for bus 10
    ppc['bus'][23, 3] = busdata[23, 6]    # Shunt Qd for bus 24 (represented as reactive load)
    ppc['bus'][23, 5] = busdata[23, 5]    # Shunt Bs for bus 24

    # Correcting bus 21 to be a PQ bus, not a slack bus
    if ppc['bus'][20, 1] == 3:
        ppc['bus'][20, 1] = 1

    # --- Generator Data ---
    # Extracted from rows in busdata where bus type is 1 (Slack) or 2 (PV)
    # Columns: bus, Pg, Qg, Qmax, Qmin, Vg, mBase, status, Pmax, Pmin
    gen_buses = busdata[np.where((busdata[:, 1] == 1) | (busdata[:, 1] == 2))]
    ppc['gen'] = np.zeros((len(gen_buses), 10))
    for i, gen_bus in enumerate(gen_buses):
        ppc['gen'][i, 0] = gen_bus[0]       # bus
        ppc['gen'][i, 1] = gen_bus[4]       # Pg
        ppc['gen'][i, 2] = gen_bus[5]       # Qg
        ppc['gen'][i, 3] = gen_bus[9]       # Qmax
        ppc['gen'][i, 4] = gen_bus[8]       # Qmin
        ppc['gen'][i, 5] = gen_bus[2]       # Vg
        ppc['gen'][i, 6] = 100              # mBase
        ppc['gen'][i, 7] = 1                # status
        ppc['gen'][i, 8] = 200              # Pmax (default, adjust as needed)
        ppc['gen'][i, 9] = 0                # Pmin


    # --- Branch Data ---
    # From user's 'linedata'
    # Columns: fbus, tbus, r, x, b, rateA, rateB, rateC, ratio, angle, status
    linedata = np.array([
        [1,  2,  0.0192, 0.0575, 0.0264, 1], 
       # [1,  3,  0.0452, 0.1652, 0.0204, 1],
        [2,  4,  0.0570, 0.1737, 0.0184, 1], 
        [3,  4,  0.0132, 0.0379, 0.0042, 1],
        [2,  5,  0.0472, 0.1983, 0.0209, 1], 
        [2,  6,  0.0581, 0.1763, 0.0187, 1],
        [4,  6,  0.0119, 0.0414, 0.0045, 1], 
        [5,  7,  0.0460, 0.1160, 0.0102, 1],
        [6,  7,  0.0267, 0.0820, 0.0085, 1], 
        [6,  8,  0.0120, 0.0420, 0.0045, 1],
        [6,  9,  0.0,    0.2080, 0.0,    0.978], 
        [6,  10, 0.0,    0.5560, 0.0,    0.969],
        [9,  11, 0.0,    0.2080, 0.0,    1], 
        [9,  10, 0.0,    0.1100, 0.0,    1],
        [4,  12, 0.0,    0.2560, 0.0,    0.932], 
        [12, 13, 0.0,    0.1400, 0.0,    1],
        [12, 14, 0.1231, 0.2559, 0.0,    1], 
        [12, 15, 0.0662, 0.1304, 0.0,    1],
        [12, 16, 0.0945, 0.1987, 0.0,    1], 
        [14, 15, 0.2210, 0.1997, 0.0,    1],
        [16, 17, 0.0824, 0.1923, 0.0,    1], 
        [15, 18, 0.1073, 0.2185, 0.0,    1],
        [18, 19, 0.0639, 0.1292, 0.0,    1], 
        [19, 20, 0.0340, 0.0680, 0.0,    1],
        [10, 20, 0.0936, 0.2090, 0.0,    1], 
        [10, 17, 0.0324, 0.0845, 0.0,    1],
        [10, 21, 0.0348, 0.0749, 0.0,    1], 
        [10, 22, 0.0727, 0.1499, 0.0,    1],
        [21, 23, 0.0116, 0.0236, 0.0,    1], 
        [15, 23, 0.1000, 0.2020, 0.0,    1],
        [22, 24, 0.1150, 0.1790, 0.0,    1], 
        [23, 24, 0.1320, 0.2700, 0.0,    1],
        [24, 25, 0.1885, 0.3292, 0.0,    1], 
        [25, 26, 0.2544, 0.3800, 0.0,    1],
        [25, 27, 0.1093, 0.2087, 0.0,    1], 
        [28, 27, 0.0,    0.3960, 0.0,    0.968],
        [27, 29, 0.2198, 0.4153, 0.0,    1], 
        [27, 30, 0.3202, 0.6027, 0.0,    1],
        [29, 30, 0.2399, 0.4533, 0.0,    1], 
        [8,  28, 0.0636, 0.2000, 0.0214, 1],
        [6,  28, 0.0169, 0.0599, 0.065,  1]
    ])
    ppc['branch'] = np.zeros((len(linedata), 11))
    ppc['branch'][:, 0:5] = linedata[:, 0:5]
    ppc['branch'][:, 8] = linedata[:, 5] # Tap ratio
    ppc['branch'][:, 10] = 1 # Status
    ppc['branch'][:, 5] = 9999 # rateA (default to a high value)


    return ppc


def run_newton_raphson_power_flow():
    """
    This function performs a Newton-Raphson power flow analysis on a
    custom 30-bus test system using the PYPOWER library.
    """

    # --- 1. Load Custom System Data ---
    ppc = create_custom_30_bus_system()

    print("--- Custom 30-Bus System Data Loaded ---")
    print(f"System Base MVA: {ppc['baseMVA']}")
    print(f"Number of buses: {len(ppc['bus'])}")
    print(f"Number of generators: {len(ppc['gen'])}")
    print(f"Number of transmission lines: {len(ppc['branch'])}")
    print("-" * 40)


    # --- 2. Set Power Flow Options ---
    ppopt = ppoption(PF_ALG=1, VERBOSE=1, OUT_ALL=0)


    # --- 3. Run the Power Flow Solver ---
    print("\n--- Running Newton-Raphson Power Flow ---")
    results, success = runpf(ppc, ppopt)
    print("Power flow calculation complete.")
    print("-" * 40)


    # --- 4. Display Results ---
    if success:
        print("\n--- Power Flow Analysis Results ---")
        print("Convergence achieved successfully!")

        print("\nBus Voltage Magnitudes and Angles:")
        print("-------------------------------------------------")
        print("Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)")
        print("-------------------------------------------------")
        for bus in results['bus']:
            bus_num = int(bus[0])
            vm_pu = bus[7]
            va_deg = bus[8]
            pd_mw = bus[2]
            qd_mvar = bus[3]
            print(f"{bus_num:<8}| {vm_pu:<16.4f}| {va_deg:<13.4f}| {pd_mw:<11.2f}| {qd_mvar:<12.2f}")
        print("-------------------------------------------------")

        print("\nGenerator Dispatch:")
        print("------------------------------------------")
        print("Gen on Bus | P_gen (MW)   | Q_gen (MVAr)")
        print("------------------------------------------")
        for gen in results['gen']:
             bus_num = int(gen[0])
             pg_mw = gen[1]
             qg_mvar = gen[2]
             print(f"{bus_num:<11}| {pg_mw:<12.2f}| {qg_mvar:<12.2f}")
        print("------------------------------------------")


    else:
        print("\n--- Power Flow Analysis Failed ---")
        print("The Newton-Raphson algorithm did not converge.")


if __name__ == '__main__':
    run_newton_raphson_power_flow()


--- Custom 30-Bus System Data Loaded ---
System Base MVA: 100.0
Number of buses: 30
Number of generators: 6
Number of transmission lines: 40
----------------------------------------

--- Running Newton-Raphson Power Flow ---
PYPOWER Version 5.1.18, 10-Apr-2025 -- AC Power Flow (Newton)


Newton's method power flow converged in 4 iterations.
Power flow calculation complete.
----------------------------------------

--- Power Flow Analysis Results ---
Convergence achieved successfully!

Bus Voltage Magnitudes and Angles:
-------------------------------------------------
Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)
-------------------------------------------------
1       | 1.0600          | 0.0000       | 0.00       | 0.00        
2       | 1.0430          | -8.4678      | 21.70      | 12.70       
3       | 0.9995          | -16.6409     | 2.40       | 1.20        
4       | 1.0002          | -16.5963     | 7.60       | 1.60        
5       | 1.0100          | 

In [11]:
# First, you need to install the PYPOWER library.
# You can do this by running the following command in your terminal:
# pip install pypower

import numpy as np
from pypower.api import runpf, ppoption
from pprint import pprint

def create_custom_30_bus_system():
    """
    This function creates a custom PYPOWER case (ppc) for a 30-bus system,
    populated with the user-provided busdata and linedata.
    """
    ppc = {
        "version": '2',
        "baseMVA": 100.0,
    }

    # --- Bus Data ---
    # Data is extracted and formatted from the user's 'busdata' matrix.
    # Bus type mapping: 1 -> 3 (Slack), 2 -> 2 (PV), 3 -> 1 (PQ)
    busdata = np.array([
        [1,  1, 1.06,  0,   0,   0,    0,    0,   0,   0],
        [2,  2, 1.043, 0,  40,  50.0, 21.7, 12.7, -40, 50],
        [3,  3, 1.0,   0,   0,   0,   2.4,  1.2,   0,   0],
        [4,  3, 1.06,  0,   0,   0,   7.6,  1.6,   0,   0],
        [5,  2, 1.01,  0,   0,  37.0, 94.2, 19.0, -40, 40],
        [6,  3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [7,  3, 1.0,   0,   0,   0,  22.8, 10.9,   0,   0],
        [8,  2, 1.01,  0,   0,  37.3, 30.0, 30.0, -10, 40],
        [9,  3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [10, 3, 1.0,   0,   0,  19.0,  5.8,  2.0,   0,   0],
        [11, 2, 1.082, 0,   0,  16.2,  0.0,  0.0,  -6,  24],
        [12, 3, 1.0,   0,   0,   0,  11.2,  7.5,   0,   0],
        [13, 2, 1.071, 0,   0,  10.6,  0.0,  0.0,  -6,  24],
        [14, 3, 1.0,   0,   0,   0,   6.2,  1.6,   0,   0],
        [15, 3, 1.0,   0,   0,   0,   8.2,  2.5,   0,   0],
        [16, 3, 1.0,   0,   0,   0,   3.5,  1.8,   0,   0],
        [17, 3, 1.0,   0,   0,   0,   9.0,  5.8,   0,   0],
        [18, 3, 1.0,   0,   0,   0,   3.2,  0.9,   0,   0],
        [19, 3, 1.0,   0,   0,   0,   9.5,  3.4,   0,   0],
        [20, 3, 1.0,   0,   0,   0,   2.2,  0.7,   0,   0],
        [21, 3, 1.0,   0,   0,   0,  17.5, 11.2,   0,   0],
        [22, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [23, 3, 1.0,   0,   0,   0,   3.2,  1.6,   0,   0],
        [24, 3, 1.0,   0,   0,  4.3,  8.7,  6.7,   0,   0],
        [25, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [26, 3, 1.0,   0,   0,   0,   3.5,  2.3,   0,   0],
        [27, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [28, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [29, 3, 1.0,   0,   0,   0,   2.4,  0.9,   0,   0],
        [30, 3, 1.0,   0,   0,   0,  10.6,  1.9,   0,   0]
    ])

    # PYPOWER bus matrix
    # Columns: bus_i, type, Pd, Qd, Gs, Bs, area, Vm, Va, baseKV, zone, Vmax, Vmin
    ppc['bus'] = np.zeros((30, 13))
    for i in range(30):
        bus_type = busdata[i, 1]
        if bus_type == 1:
            ppc['bus'][i, 1] = 3  # Slack
        elif bus_type == 2:
            ppc['bus'][i, 1] = 2  # PV
        else:
            ppc['bus'][i, 1] = 1  # PQ

        ppc['bus'][i, 0] = busdata[i, 0]    # Bus Number
        ppc['bus'][i, 2] = busdata[i, 6]    # Pd
        ppc['bus'][i, 3] = busdata[i, 7]    # Qd
        ppc['bus'][i, 7] = busdata[i, 2]    # Vm
        ppc['bus'][i, 8] = busdata[i, 3]    # Va
        # Set default values for other fields
        ppc['bus'][i, 6] = 1                # area
        ppc['bus'][i, 9] = 132              # baseKV
        ppc['bus'][i, 10] = 1               # zone
        ppc['bus'][i, 11] = 1.1             # Vmax
        ppc['bus'][i, 12] = 0.9             # Vmin

    # Special handling for shunts on buses 10 and 24 from busdata
    ppc['bus'][9, 5] = busdata[9, 5]     # Shunt Bs for bus 10
    ppc['bus'][23, 3] = busdata[23, 6]    # Shunt Qd for bus 24 (represented as reactive load)
    ppc['bus'][23, 5] = busdata[23, 5]    # Shunt Bs for bus 24

    # Correcting bus 21 to be a PQ bus, not a slack bus
    if ppc['bus'][20, 1] == 3:
        ppc['bus'][20, 1] = 1

    # --- Generator Data ---
    # Extracted from rows in busdata where bus type is 1 (Slack) or 2 (PV)
    # Columns: bus, Pg, Qg, Qmax, Qmin, Vg, mBase, status, Pmax, Pmin
    gen_buses = busdata[np.where((busdata[:, 1] == 1) | (busdata[:, 1] == 2))]
    ppc['gen'] = np.zeros((len(gen_buses), 10))
    for i, gen_bus in enumerate(gen_buses):
        ppc['gen'][i, 0] = gen_bus[0]       # bus
        ppc['gen'][i, 1] = gen_bus[4]       # Pg
        ppc['gen'][i, 2] = gen_bus[5]       # Qg
        ppc['gen'][i, 3] = gen_bus[9]       # Qmax
        ppc['gen'][i, 4] = gen_bus[8]       # Qmin
        ppc['gen'][i, 5] = gen_bus[2]       # Vg
        ppc['gen'][i, 6] = 100              # mBase
        ppc['gen'][i, 7] = 1                # status
        ppc['gen'][i, 8] = 200              # Pmax (default, adjust as needed)
        ppc['gen'][i, 9] = 0                # Pmin


    # --- Branch Data ---
    # From user's 'linedata'
    # Columns: fbus, tbus, r, x, b, rateA, rateB, rateC, ratio, angle, status
    linedata = np.array([
        [1,  2,  0.0192, 0.0575, 0.0264, 1], 
        [1,  3,  0.0452, 0.1652, 0.0204, 1],
        [2,  4,  0.0570, 0.1737, 0.0184, 1], 
        [3,  4,  0.0132, 0.0379, 0.0042, 1],
        [2,  5,  0.0472, 0.1983, 0.0209, 1], 
        [2,  6,  0.0581, 0.1763, 0.0187, 1],
        [4,  6,  0.0119, 0.0414, 0.0045, 1], 
        [5,  7,  0.0460, 0.1160, 0.0102, 1],
        [6,  7,  0.0267, 0.0820, 0.0085, 1], 
        [6,  8,  0.0120, 0.0420, 0.0045, 1],
        [6,  9,  0.0,    0.2080, 0.0,    0.978], 
        [6,  10, 0.0,    0.5560, 0.0,    0.969],
        [9,  11, 0.0,    0.2080, 0.0,    1], 
        [9,  10, 0.0,    0.1100, 0.0,    1],
        [4,  12, 0.0,    0.2560, 0.0,    0.932], 
        [12, 13, 0.0,    0.1400, 0.0,    1],
        [12, 14, 0.1231, 0.2559, 0.0,    1], 
        [12, 15, 0.0662, 0.1304, 0.0,    1],
        [12, 16, 0.0945, 0.1987, 0.0,    1], 
        [14, 15, 0.2210, 0.1997, 0.0,    1],
        [16, 17, 0.0824, 0.1923, 0.0,    1], 
        [15, 18, 0.1073, 0.2185, 0.0,    1],
        [18, 19, 0.0639, 0.1292, 0.0,    1], 
        [19, 20, 0.0340, 0.0680, 0.0,    1],
        [10, 20, 0.0936, 0.2090, 0.0,    1], 
        [10, 17, 0.0324, 0.0845, 0.0,    1],
        [10, 21, 0.0348, 0.0749, 0.0,    1], 
        [10, 22, 0.0727, 0.1499, 0.0,    1],
        [21, 23, 0.0116, 0.0236, 0.0,    1], 
        [15, 23, 0.1000, 0.2020, 0.0,    1],
        [22, 24, 0.1150, 0.1790, 0.0,    1], 
        [23, 24, 0.1320, 0.2700, 0.0,    1],
        [24, 25, 0.1885, 0.3292, 0.0,    1], 
        [25, 26, 0.2544, 0.3800, 0.0,    1],
        [25, 27, 0.1093, 0.2087, 0.0,    1], 
        [28, 27, 0.0,    0.3960, 0.0,    0.968],
        [27, 29, 0.2198, 0.4153, 0.0,    1], 
        [27, 30, 0.3202, 0.6027, 0.0,    1],
        [29, 30, 0.2399, 0.4533, 0.0,    1], 
        [8,  28, 0.0636, 0.2000, 0.0214, 1],
        [6,  28, 0.0169, 0.0599, 0.065,  1]
    ])
    ppc['branch'] = np.zeros((len(linedata), 11))
    ppc['branch'][:, 0:5] = linedata[:, 0:5]
    ppc['branch'][:, 8] = linedata[:, 5] # Tap ratio
    ppc['branch'][:, 10] = 1 # Status
    ppc['branch'][:, 5] = 9999 # rateA (default to a high value)


    return ppc


def run_newton_raphson_power_flow():
    """
    This function performs a Newton-Raphson power flow analysis on a
    custom 30-bus test system using the PYPOWER library.
    """

    # --- 1. Load Custom System Data ---
    ppc = create_custom_30_bus_system()

    print("--- Custom 30-Bus System Data Loaded ---")
    print(f"System Base MVA: {ppc['baseMVA']}")
    print(f"Number of buses: {len(ppc['bus'])}")
    print(f"Number of generators: {len(ppc['gen'])}")
    print(f"Number of transmission lines: {len(ppc['branch'])}")
    print("-" * 40)


    # --- 2. Set Power Flow Options ---
    ppopt = ppoption(PF_ALG=1, VERBOSE=1, OUT_ALL=0)


    # --- 3. Run the Power Flow Solver ---
    print("\n--- Running Newton-Raphson Power Flow ---")
    results, success = runpf(ppc, ppopt)
    print("Power flow calculation complete.")
    print("-" * 40)


    # --- 4. Display Results ---
    if success:
        print("\n--- Power Flow Analysis Results ---")
        print("Convergence achieved successfully!")

        print("\nBus Voltage Magnitudes and Angles:")
        print("-------------------------------------------------")
        print("Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)")
        print("-------------------------------------------------")
        for bus in results['bus']:
            bus_num = int(bus[0])
            vm_pu = bus[7]
            va_deg = bus[8]
            pd_mw = bus[2]
            qd_mvar = bus[3]
            print(f"{bus_num:<8}| {vm_pu:<16.4f}| {va_deg:<13.4f}| {pd_mw:<11.2f}| {qd_mvar:<12.2f}")
        print("-------------------------------------------------")

        print("\nGenerator Dispatch:")
        print("------------------------------------------")
        print("Gen on Bus | P_gen (MW)   | Q_gen (MVAr)")
        print("------------------------------------------")
        for gen in results['gen']:
             bus_num = int(gen[0])
             pg_mw = gen[1]
             qg_mvar = gen[2]
             print(f"{bus_num:<11}| {pg_mw:<12.2f}| {qg_mvar:<12.2f}")
        print("------------------------------------------")


    else:
        print("\n--- Power Flow Analysis Failed ---")
        print("The Newton-Raphson algorithm did not converge.")


if __name__ == '__main__':
    run_newton_raphson_power_flow()


--- Custom 30-Bus System Data Loaded ---
System Base MVA: 100.0
Number of buses: 30
Number of generators: 6
Number of transmission lines: 41
----------------------------------------

--- Running Newton-Raphson Power Flow ---
PYPOWER Version 5.1.18, 10-Apr-2025 -- AC Power Flow (Newton)


Newton's method power flow converged in 4 iterations.
Power flow calculation complete.
----------------------------------------

--- Power Flow Analysis Results ---
Convergence achieved successfully!

Bus Voltage Magnitudes and Angles:
-------------------------------------------------
Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)
-------------------------------------------------
1       | 1.0600          | 0.0000       | 0.00       | 0.00        
2       | 1.0430          | -5.3537      | 21.70      | 12.70       
3       | 1.0196          | -7.5191      | 2.40       | 1.20        
4       | 1.0109          | -9.2780      | 7.60       | 1.60        
5       | 1.0100          | 

In [13]:
# First, you need to install the PYPOWER library.
# You can do this by running the following command in your terminal:
# pip install pypower

import numpy as np
from pypower.api import runpf, ppoption
from pprint import pprint

def create_custom_30_bus_system():
    """
    This function creates a custom PYPOWER case (ppc) for a 30-bus system,
    populated with the user-provided busdata and linedata.
    """
    ppc = {
        "version": '2',
        "baseMVA": 100.0,
    }

    # --- Bus Data ---
    # Data is extracted and formatted from the user's 'busdata' matrix.
    # Bus type mapping: 1 -> 3 (Slack), 2 -> 2 (PV), 3 -> 1 (PQ)
    busdata = np.array([
        [1,  1, 1.06,  0,   0,   0,    0,    0,   0,   0],
        [2,  2, 1.043, 0,  40,  50.0, 21.7, 12.7, -40, 50],
        [3,  3, 1.0,   0,   0,   0,   2.4,  1.2,   0,   0],
        [4,  3, 1.06,  0,   0,   0,   7.6,  1.6,   0,   0],
        [5,  2, 1.01,  0,   0,  37.0, 94.2, 19.0, -40, 40],
        [6,  3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [7,  3, 1.0,   0,   0,   0,  22.8, 10.9,   0,   0],
        [8,  2, 1.01,  0,   0,  37.3, 30.0, 30.0, -10, 40],
        [9,  3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [10, 3, 1.0,   0,   0,  19.0,  5.8,  2.0,   0,   0],
        [11, 2, 1.082, 0,   0,  16.2,  0.0,  0.0,  -6,  24],
        [12, 3, 1.0,   0,   0,   0,  11.2,  7.5,   0,   0],
        [13, 2, 1.071, 0,   0,  10.6,  0.0,  0.0,  -6,  24],
        [14, 3, 1.0,   0,   0,   0,   6.2,  1.6,   0,   0],
        [15, 3, 1.0,   0,   0,   0,   8.2,  2.5,   0,   0],
        [16, 3, 1.0,   0,   0,   0,   3.5,  1.8,   0,   0],
        [17, 3, 1.0,   0,   0,   0,   9.0,  5.8,   0,   0],
        [18, 3, 1.0,   0,   0,   0,   3.2,  0.9,   0,   0],
        [19, 3, 1.0,   0,   0,   0,   9.5,  3.4,   0,   0],
        [20, 3, 1.0,   0,   0,   0,   2.2,  0.7,   0,   0],
        [21, 3, 1.0,   0,   0,   0,  17.5, 11.2,   0,   0],
        [22, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [23, 3, 1.0,   0,   0,   0,   3.2,  1.6,   0,   0],
        [24, 3, 1.0,   0,   0,  4.3,  8.7,  6.7,   0,   0],
        [25, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [26, 3, 1.0,   0,   0,   0,   3.5,  2.3,   0,   0],
        [27, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [28, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [29, 3, 1.0,   0,   0,   0,   2.4,  0.9,   0,   0],
        [30, 3, 1.0,   0,   0,   0,  10.6,  1.9,   0,   0]
    ])

    # PYPOWER bus matrix
    # Columns: bus_i, type, Pd, Qd, Gs, Bs, area, Vm, Va, baseKV, zone, Vmax, Vmin
    ppc['bus'] = np.zeros((30, 13))
    for i in range(30):
        bus_type = busdata[i, 1]
        if bus_type == 1:
            ppc['bus'][i, 1] = 3  # Slack
        elif bus_type == 2:
            ppc['bus'][i, 1] = 2  # PV
        else:
            ppc['bus'][i, 1] = 1  # PQ

        ppc['bus'][i, 0] = busdata[i, 0]    # Bus Number
        ppc['bus'][i, 2] = busdata[i, 6]    # Pd
        ppc['bus'][i, 3] = busdata[i, 7]    # Qd
        ppc['bus'][i, 7] = busdata[i, 2]    # Vm
        ppc['bus'][i, 8] = busdata[i, 3]    # Va
        # Set default values for other fields
        ppc['bus'][i, 6] = 1                # area
        ppc['bus'][i, 9] = 132              # baseKV
        ppc['bus'][i, 10] = 1               # zone
        ppc['bus'][i, 11] = 1.1             # Vmax
        ppc['bus'][i, 12] = 0.9             # Vmin

    # Special handling for shunts on buses 10 and 24 from busdata
    ppc['bus'][9, 5] = busdata[9, 5]     # Shunt Bs for bus 10
    ppc['bus'][23, 3] = busdata[23, 6]    # Shunt Qd for bus 24 (represented as reactive load)
    ppc['bus'][23, 5] = busdata[23, 5]    # Shunt Bs for bus 24

    # Correcting bus 21 to be a PQ bus, not a slack bus
    if ppc['bus'][20, 1] == 3:
        ppc['bus'][20, 1] = 1

    # --- Generator Data ---
    # Extracted from rows in busdata where bus type is 1 (Slack) or 2 (PV)
    # Columns: bus, Pg, Qg, Qmax, Qmin, Vg, mBase, status, Pmax, Pmin
    gen_buses = busdata[np.where((busdata[:, 1] == 1) | (busdata[:, 1] == 2))]
    ppc['gen'] = np.zeros((len(gen_buses), 10))
    for i, gen_bus in enumerate(gen_buses):
        ppc['gen'][i, 0] = gen_bus[0]       # bus
        ppc['gen'][i, 1] = gen_bus[4]       # Pg
        ppc['gen'][i, 2] = gen_bus[5]       # Qg
        ppc['gen'][i, 3] = gen_bus[9]       # Qmax
        ppc['gen'][i, 4] = gen_bus[8]       # Qmin
        ppc['gen'][i, 5] = gen_bus[2]       # Vg
        ppc['gen'][i, 6] = 100              # mBase
        ppc['gen'][i, 7] = 1                # status
        ppc['gen'][i, 8] = 200              # Pmax (default, adjust as needed)
        ppc['gen'][i, 9] = 0                # Pmin


    # --- Branch Data ---
    # From user's 'linedata'
    # Columns: fbus, tbus, r, x, b, rateA, rateB, rateC, ratio, angle, status
    linedata = np.array([
        [1,  2,  0.0192, 0.0575, 0.0264, 1], [1,  3,  0.0452, 0.1652, 0.0204, 1],
        [2,  4,  0.0570, 0.1737, 0.0184, 1], [3,  4,  0.0132, 0.0379, 0.0042, 1],
        [2,  5,  0.0472, 0.1983, 0.0209, 1], [2,  6,  0.0581, 0.1763, 0.0187, 1],
        [4,  6,  0.0119, 0.0414, 0.0045, 1], [5,  7,  0.0460, 0.1160, 0.0102, 1],
        [6,  7,  0.0267, 0.0820, 0.0085, 1], [6,  8,  0.0120, 0.0420, 0.0045, 1],
        [6,  9,  0.0,    0.2080, 0.0,    0.978], [6,  10, 0.0,    0.5560, 0.0,    0.969],
        [9,  11, 0.0,    0.2080, 0.0,    1], [9,  10, 0.0,    0.1100, 0.0,    1],
        [4,  12, 0.0,    0.2560, 0.0,    0.932], [12, 13, 0.0,    0.1400, 0.0,    1],
        [12, 14, 0.1231, 0.2559, 0.0,    1], [12, 15, 0.0662, 0.1304, 0.0,    1],
        [12, 16, 0.0945, 0.1987, 0.0,    1], [14, 15, 0.2210, 0.1997, 0.0,    1],
        [16, 17, 0.0824, 0.1923, 0.0,    1], [15, 18, 0.1073, 0.2185, 0.0,    1],
        [18, 19, 0.0639, 0.1292, 0.0,    1], [19, 20, 0.0340, 0.0680, 0.0,    1],
        [10, 20, 0.0936, 0.2090, 0.0,    1], [10, 17, 0.0324, 0.0845, 0.0,    1],
        [10, 21, 0.0348, 0.0749, 0.0,    1], [10, 22, 0.0727, 0.1499, 0.0,    1],
        [21, 23, 0.0116, 0.0236, 0.0,    1], [15, 23, 0.1000, 0.2020, 0.0,    1],
        [22, 24, 0.1150, 0.1790, 0.0,    1], [23, 24, 0.1320, 0.2700, 0.0,    1],
        [24, 25, 0.1885, 0.3292, 0.0,    1], [25, 26, 0.2544, 0.3800, 0.0,    1],
        [25, 27, 0.1093, 0.2087, 0.0,    1], [28, 27, 0.0,    0.3960, 0.0,    0.968],
        [27, 29, 0.2198, 0.4153, 0.0,    1], [27, 30, 0.3202, 0.6027, 0.0,    1],
        [29, 30, 0.2399, 0.4533, 0.0,    1], [8,  28, 0.0636, 0.2000, 0.0214, 1],
        [6,  28, 0.0169, 0.0599, 0.065,  1]
    ])
    ppc['branch'] = np.zeros((len(linedata), 11))
    ppc['branch'][:, 0:5] = linedata[:, 0:5]
    ppc['branch'][:, 8] = linedata[:, 5] # Tap ratio
    ppc['branch'][:, 10] = 1 # Status
    ppc['branch'][:, 5] = 9999 # rateA (default to a high value)


    return ppc


def run_newton_raphson_power_flow():
    """
    This function performs a Newton-Raphson power flow analysis on a
    custom 30-bus test system using the PYPOWER library.
    """

    # --- 1. Load Custom System Data ---
    ppc = create_custom_30_bus_system()

    print("--- Custom 30-Bus System Data Loaded ---")
    print(f"System Base MVA: {ppc['baseMVA']}")
    print(f"Number of buses: {len(ppc['bus'])}")
    print(f"Number of generators: {len(ppc['gen'])}")
    print(f"Number of transmission lines: {len(ppc['branch'])}")
    print("-" * 40)


    # --- 2. Set Power Flow Options ---
    ppopt = ppoption(PF_ALG=1, VERBOSE=1, OUT_ALL=0)


    # --- 3. Run the Power Flow Solver ---
    print("\n--- Running Newton-Raphson Power Flow ---")
    results, success = runpf(ppc, ppopt)
    print("Power flow calculation complete.")
    print("-" * 40)


    # --- 4. Display Results ---
    if success:
        print("\n--- Power Flow Analysis Results ---")
        print("Convergence achieved successfully!")

        # --- Bus Voltages ---
        print("\nBus Voltage Magnitudes and Angles:")
        print("-------------------------------------------------")
        print("Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)")
        print("-------------------------------------------------")
        for bus in results['bus']:
            bus_num = int(bus[0])
            vm_pu = bus[7]
            va_deg = bus[8]
            pd_mw = bus[2]
            qd_mvar = bus[3]
            print(f"{bus_num:<8}| {vm_pu:<16.4f}| {va_deg:<13.4f}| {pd_mw:<11.2f}| {qd_mvar:<12.2f}")
        print("-------------------------------------------------")

        # --- Generator Dispatch ---
        print("\nGenerator Dispatch:")
        print("------------------------------------------")
        print("Gen on Bus | P_gen (MW)   | Q_gen (MVAr)")
        print("------------------------------------------")
        for gen in results['gen']:
             bus_num = int(gen[0])
             pg_mw = gen[1]
             qg_mvar = gen[2]
             print(f"{bus_num:<11}| {pg_mw:<12.2f}| {qg_mvar:<12.2f}")
        print("------------------------------------------")

        # --- Branch Power Flows ---
        # The results['branch'] matrix contains the power flow results.
        # Columns 13, 14: Real (PF) and reactive (QF) power flow at the "from" bus end
        # Columns 15, 16: Real (PT) and reactive (QT) power flow at the "to" bus end
        print("\nBranch Power Flows:")
        print("--------------------------------------------------------------------------------")
        print("From Bus | To Bus | P_from (MW) | Q_from (MVAr) | P_to (MW)   | Q_to (MVAr)")
        print("--------------------------------------------------------------------------------")
        for branch in results['branch']:
            f_bus = int(branch[0])
            t_bus = int(branch[1])
            p_from = branch[13]
            q_from = branch[14]
            p_to = branch[15]
            q_to = branch[16]
            print(f"{f_bus:<9}| {t_bus:<7}| {p_from:<12.4f}| {q_from:<14.4f}| {p_to:<12.4f}| {q_to:<11.4f}")
        print("--------------------------------------------------------------------------------")

    else:
        print("\n--- Power Flow Analysis Failed ---")
        print("The Newton-Raphson algorithm did not converge.")


if __name__ == '__main__':
    run_newton_raphson_power_flow()

--- Custom 30-Bus System Data Loaded ---
System Base MVA: 100.0
Number of buses: 30
Number of generators: 6
Number of transmission lines: 41
----------------------------------------

--- Running Newton-Raphson Power Flow ---
PYPOWER Version 5.1.18, 10-Apr-2025 -- AC Power Flow (Newton)


Newton's method power flow converged in 4 iterations.
Power flow calculation complete.
----------------------------------------

--- Power Flow Analysis Results ---
Convergence achieved successfully!

Bus Voltage Magnitudes and Angles:
-------------------------------------------------
Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)
-------------------------------------------------
1       | 1.0600          | 0.0000       | 0.00       | 0.00        
2       | 1.0430          | -5.3537      | 21.70      | 12.70       
3       | 1.0196          | -7.5191      | 2.40       | 1.20        
4       | 1.0109          | -9.2780      | 7.60       | 1.60        
5       | 1.0100          | 

In [15]:
# First, you need to install the PYPOWER library.
# You can do this by running the following command in your terminal:
# pip install pypower

import numpy as np
from pypower.api import runpf, ppoption
from pprint import pprint

def create_custom_30_bus_system():
    """
    This function creates a custom PYPOWER case (ppc) for a 30-bus system,
    populated with the user-provided busdata and linedata.
    """
    ppc = {
        "version": '2',
        "baseMVA": 100.0,
    }

    # --- Bus Data ---
    # Data is extracted and formatted from the user's 'busdata' matrix.
    # Bus type mapping: 1 -> 3 (Slack), 2 -> 2 (PV), 3 -> 1 (PQ)
    busdata = np.array([
        [1,  1, 1.06,  0,   0,   0,    0,    0,   0,   0],
        [2,  2, 1.043, 0,  40,  50.0, 21.7, 12.7, -40, 50],
        [3,  3, 1.0,   0,   0,   0,   2.4,  1.2,   0,   0],
        [4,  3, 1.06,  0,   0,   0,   7.6,  1.6,   0,   0],
        [5,  2, 1.01,  0,   0,  37.0, 94.2, 19.0, -40, 40],
        [6,  3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [7,  3, 1.0,   0,   0,   0,  22.8, 10.9,   0,   0],
        [8,  2, 1.01,  0,   0,  37.3, 30.0, 30.0, -10, 40],
        [9,  3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [10, 3, 1.0,   0,   0,  19.0,  5.8,  2.0,   0,   0],
        [11, 2, 1.082, 0,   0,  16.2,  0.0,  0.0,  -6,  24],
        [12, 3, 1.0,   0,   0,   0,  11.2,  7.5,   0,   0],
        [13, 2, 1.071, 0,   0,  10.6,  0.0,  0.0,  -6,  24],
        [14, 3, 1.0,   0,   0,   0,   6.2,  1.6,   0,   0],
        [15, 3, 1.0,   0,   0,   0,   8.2,  2.5,   0,   0],
        [16, 3, 1.0,   0,   0,   0,   3.5,  1.8,   0,   0],
        [17, 3, 1.0,   0,   0,   0,   9.0,  5.8,   0,   0],
        [18, 3, 1.0,   0,   0,   0,   3.2,  0.9,   0,   0],
        [19, 3, 1.0,   0,   0,   0,   9.5,  3.4,   0,   0],
        [20, 3, 1.0,   0,   0,   0,   2.2,  0.7,   0,   0],
        [21, 3, 1.0,   0,   0,   0,  17.5, 11.2,   0,   0],
        [22, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [23, 3, 1.0,   0,   0,   0,   3.2,  1.6,   0,   0],
        [24, 3, 1.0,   0,   0,  4.3,  8.7,  6.7,   0,   0],
        [25, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [26, 3, 1.0,   0,   0,   0,   3.5,  2.3,   0,   0],
        [27, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [28, 3, 1.0,   0,   0,   0,   0.0,  0.0,   0,   0],
        [29, 3, 1.0,   0,   0,   0,   2.4,  0.9,   0,   0],
        [30, 3, 1.0,   0,   0,   0,  10.6,  1.9,   0,   0]
    ])

    # PYPOWER bus matrix
    # Columns: bus_i, type, Pd, Qd, Gs, Bs, area, Vm, Va, baseKV, zone, Vmax, Vmin
    ppc['bus'] = np.zeros((30, 13))
    for i in range(30):
        bus_type = busdata[i, 1]
        if bus_type == 1:
            ppc['bus'][i, 1] = 3  # Slack
        elif bus_type == 2:
            ppc['bus'][i, 1] = 2  # PV
        else:
            ppc['bus'][i, 1] = 1  # PQ

        ppc['bus'][i, 0] = busdata[i, 0]    # Bus Number
        ppc['bus'][i, 2] = busdata[i, 6]    # Pd
        ppc['bus'][i, 3] = busdata[i, 7]    # Qd
        ppc['bus'][i, 7] = busdata[i, 2]    # Vm
        ppc['bus'][i, 8] = busdata[i, 3]    # Va
        # Set default values for other fields
        ppc['bus'][i, 6] = 1                # area
        ppc['bus'][i, 9] = 132              # baseKV
        ppc['bus'][i, 10] = 1               # zone
        ppc['bus'][i, 11] = 1.1             # Vmax
        ppc['bus'][i, 12] = 0.9             # Vmin

    # Special handling for shunts on buses 10 and 24 from busdata
    ppc['bus'][9, 5] = busdata[9, 5]     # Shunt Bs for bus 10
    ppc['bus'][23, 3] = busdata[23, 6]    # Shunt Qd for bus 24 (represented as reactive load)
    ppc['bus'][23, 5] = busdata[23, 5]    # Shunt Bs for bus 24

    # Correcting bus 21 to be a PQ bus, not a slack bus
    if ppc['bus'][20, 1] == 3:
        ppc['bus'][20, 1] = 1

    # --- Generator Data ---
    # Extracted from rows in busdata where bus type is 1 (Slack) or 2 (PV)
    # Columns: bus, Pg, Qg, Qmax, Qmin, Vg, mBase, status, Pmax, Pmin
    gen_buses = busdata[np.where((busdata[:, 1] == 1) | (busdata[:, 1] == 2))]
    ppc['gen'] = np.zeros((len(gen_buses), 10))
    for i, gen_bus in enumerate(gen_buses):
        ppc['gen'][i, 0] = gen_bus[0]       # bus
        ppc['gen'][i, 1] = gen_bus[4]       # Pg
        ppc['gen'][i, 2] = gen_bus[5]       # Qg
        ppc['gen'][i, 3] = gen_bus[9]       # Qmax
        ppc['gen'][i, 4] = gen_bus[8]       # Qmin
        ppc['gen'][i, 5] = gen_bus[2]       # Vg
        ppc['gen'][i, 6] = 100              # mBase
        ppc['gen'][i, 7] = 1                # status
        ppc['gen'][i, 8] = 200              # Pmax (default, adjust as needed)
        ppc['gen'][i, 9] = 0                # Pmin


    # --- Branch Data ---
    # From user's 'linedata'
    # Columns: fbus, tbus, r, x, b, rateA, rateB, rateC, ratio, angle, status
    linedata = np.array([
        [1,  2,  0.0192, 0.0575, 0.0264, 1], [1,  3,  0.0452, 0.1652, 0.0204, 1],
        [2,  4,  0.0570, 0.1737, 0.0184, 1], [3,  4,  0.0132, 0.0379, 0.0042, 1],
        [2,  5,  0.0472, 0.1983, 0.0209, 1], [2,  6,  0.0581, 0.1763, 0.0187, 1],
        [4,  6,  0.0119, 0.0414, 0.0045, 1], [5,  7,  0.0460, 0.1160, 0.0102, 1],
        [6,  7,  0.0267, 0.0820, 0.0085, 1], [6,  8,  0.0120, 0.0420, 0.0045, 1],
        [6,  9,  0.0,    0.2080, 0.0,    0.978], [6,  10, 0.0,    0.5560, 0.0,    0.969],
        [9,  11, 0.0,    0.2080, 0.0,    1], [9,  10, 0.0,    0.1100, 0.0,    1],
        [4,  12, 0.0,    0.2560, 0.0,    0.932], [12, 13, 0.0,    0.1400, 0.0,    1],
        [12, 14, 0.1231, 0.2559, 0.0,    1], [12, 15, 0.0662, 0.1304, 0.0,    1],
        [12, 16, 0.0945, 0.1987, 0.0,    1], [14, 15, 0.2210, 0.1997, 0.0,    1],
        [16, 17, 0.0824, 0.1923, 0.0,    1], [15, 18, 0.1073, 0.2185, 0.0,    1],
        [18, 19, 0.0639, 0.1292, 0.0,    1], [19, 20, 0.0340, 0.0680, 0.0,    1],
        [10, 20, 0.0936, 0.2090, 0.0,    1], [10, 17, 0.0324, 0.0845, 0.0,    1],
        [10, 21, 0.0348, 0.0749, 0.0,    1], [10, 22, 0.0727, 0.1499, 0.0,    1],
        [21, 23, 0.0116, 0.0236, 0.0,    1], [15, 23, 0.1000, 0.2020, 0.0,    1],
        [22, 24, 0.1150, 0.1790, 0.0,    1], [23, 24, 0.1320, 0.2700, 0.0,    1],
        [24, 25, 0.1885, 0.3292, 0.0,    1], [25, 26, 0.2544, 0.3800, 0.0,    1],
        [25, 27, 0.1093, 0.2087, 0.0,    1], [28, 27, 0.0,    0.3960, 0.0,    0.968],
        [27, 29, 0.2198, 0.4153, 0.0,    1], [27, 30, 0.3202, 0.6027, 0.0,    1],
        [29, 30, 0.2399, 0.4533, 0.0,    1], [8,  28, 0.0636, 0.2000, 0.0214, 1],
        [6,  28, 0.0169, 0.0599, 0.065,  1]
    ])
    ppc['branch'] = np.zeros((len(linedata), 11))
    ppc['branch'][:, 0:5] = linedata[:, 0:5]
    ppc['branch'][:, 8] = linedata[:, 5] # Tap ratio
    ppc['branch'][:, 10] = 1 # Status
    ppc['branch'][:, 5] = 9999 # rateA (default to a high value)


    return ppc


def run_newton_raphson_power_flow():
    """
    This function performs a Newton-Raphson power flow analysis on a
    custom 30-bus test system using the PYPOWER library.
    """

    # --- 1. Load Custom System Data ---
    ppc = create_custom_30_bus_system()

    print("--- Custom 30-Bus System Data Loaded ---")
    print(f"System Base MVA: {ppc['baseMVA']}")
    print(f"Number of buses: {len(ppc['bus'])}")
    print(f"Number of generators: {len(ppc['gen'])}")
    print(f"Number of transmission lines: {len(ppc['branch'])}")
    print("-" * 40)


    # --- 2. Set Power Flow Options ---
    ppopt = ppoption(PF_ALG=1, VERBOSE=1, OUT_ALL=0)


    # --- 3. Run the Power Flow Solver ---
    print("\n--- Running Newton-Raphson Power Flow ---")
    results, success = runpf(ppc, ppopt)
    print("Power flow calculation complete.")
    print("-" * 40)


    # --- 4. Display Results ---
    if success:
        print("\n--- Power Flow Analysis Results ---")
        print("Convergence achieved successfully!")

        # --- Bus Voltages ---
        print("\nBus Voltage Magnitudes and Angles:")
        print("-------------------------------------------------")
        print("Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)")
        print("-------------------------------------------------")
        for bus in results['bus']:
            bus_num = int(bus[0])
            vm_pu = bus[7]
            va_deg = bus[8]
            pd_mw = bus[2]
            qd_mvar = bus[3]
            print(f"{bus_num:<8}| {vm_pu:<16.4f}| {va_deg:<13.4f}| {pd_mw:<11.2f}| {qd_mvar:<12.2f}")
        print("-------------------------------------------------")

        # --- Generator Dispatch ---
        print("\nGenerator Dispatch:")
        print("------------------------------------------")
        print("Gen on Bus | P_gen (MW)   | Q_gen (MVAr)")
        print("------------------------------------------")
        for gen in results['gen']:
             bus_num = int(gen[0])
             pg_mw = gen[1]
             qg_mvar = gen[2]
             print(f"{bus_num:<11}| {pg_mw:<12.2f}| {qg_mvar:<12.2f}")
        print("------------------------------------------")

        # --- Branch Power Flows and Losses ---
        # This section now shows the flow IN each line and the associated losses.
        print("\nLine Power Flow and Losses:")
        print("------------------------------------------------------------------------------------------")
        print("From | To   | P Flow (MW) | Q Flow (MVAr) | S Flow (MVA) | P Loss (MW) | Q Loss (MVAr)")
        print("------------------------------------------------------------------------------------------")
        for branch in results['branch']:
            f_bus = int(branch[0])
            t_bus = int(branch[1])
            p_from = branch[13] # Real power flow at 'from' bus
            q_from = branch[14] # Reactive power flow at 'from' bus
            p_to = branch[15]   # Real power flow at 'to' bus
            q_to = branch[16]   # Reactive power flow at 'to' bus

            # Apparent power flow at the 'from' end of the line
            s_flow = np.sqrt(p_from**2 + q_from**2)

            # Power loss in the line
            # Loss = Flow_in - Flow_out. Since p_to is power INTO the 'to' bus,
            # it's negative for a positive flow. So we add them.
            p_loss = p_from + p_to
            q_loss = q_from + q_to

            print(f"{f_bus:<5}| {t_bus:<5}| {p_from:<12.4f}| {q_from:<14.4f}| {s_flow:<13.4f}| {p_loss:<12.4f}| {q_loss:<12.4f}")
        print("------------------------------------------------------------------------------------------")


    else:
        print("\n--- Power Flow Analysis Failed ---")
        print("The Newton-Raphson algorithm did not converge.")


if __name__ == '__main__':
    run_newton_raphson_power_flow()


--- Custom 30-Bus System Data Loaded ---
System Base MVA: 100.0
Number of buses: 30
Number of generators: 6
Number of transmission lines: 41
----------------------------------------

--- Running Newton-Raphson Power Flow ---
PYPOWER Version 5.1.18, 10-Apr-2025 -- AC Power Flow (Newton)


Newton's method power flow converged in 4 iterations.
Power flow calculation complete.
----------------------------------------

--- Power Flow Analysis Results ---
Convergence achieved successfully!

Bus Voltage Magnitudes and Angles:
-------------------------------------------------
Bus No. | Voltage (p.u.) | Angle (deg)   | P_load (MW) | Q_load (MVAr)
-------------------------------------------------
1       | 1.0600          | 0.0000       | 0.00       | 0.00        
2       | 1.0430          | -5.3537      | 21.70      | 12.70       
3       | 1.0196          | -7.5191      | 2.40       | 1.20        
4       | 1.0109          | -9.2780      | 7.60       | 1.60        
5       | 1.0100          | 